In [ ]:
# Enviroment Set-up
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
#!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Kaggle dataset download

!kaggle datasets download -d arnaudeq/cats-vs-dogs-5000

In [ ]:
import zipfile

with zipfile.ZipFile("cats-vs-dogs-5000.zip", 'r') as zip_ref:
    zip_ref.extractall("catsvsdogs-data")

In [ ]:
# Import Required Libraries

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

In [ ]:
# Generators (used for processing the image files in batches)

# https://keras.io/api/data_loading/image/

train_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/catsvsdogs-data/dogs_cats_sample_5000/train',
    labels="inferred",
    label_mode="int",
    class_names=None,
    batch_size=32,
    image_size=(256, 256)
    )

validation_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/catsvsdogs-data/dogs_cats_sample_5000/valid',
    labels="inferred",
    label_mode="int",
    class_names=None,
    batch_size=32,
    image_size=(256, 256)
    )

In [ ]:
# Normalizing the pixels data
def process(image, label):
    image = tf.cast(image/255. ,tf.float32)
    return image, label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)


In [ ]:
# Create a CNN Model

# Convolution Layer: 3 layers with 32 filter, 64 filter, 128 filters respectively
# Fully Connected Layer: 3 layers with 128, 64 and 1 neurons
model = Sequential()

model.add(Conv2D(32, kernel_size=(3, 3), padding='valid', activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(64, kernel_size=(3, 3), padding='valid', activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3, 3), padding='valid', activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.summary()

In [ ]:
# compiling and validating the model

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(train_ds, epochs=10, validation_data=validation_ds)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], color='red', label='train')
plt.plot(history.history['val_accuracy'], color='blue', label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'], color='red', label='train')
plt.plot(history.history['val_loss'], color='blue', label='validation')
plt.legend()
plt.show()

This indicates that the model is overfitting and needs more generalization.

Ways to reduce overfitting:
- Add more data
- Data Augmentation
- L1/L2 Regularization
- Dropout
- Batch Normalization

In [ ]:
# Testing the data on new images

import cv2

In [ ]:
test_img = cv2.imread('/content/cat.jpg')

In [ ]:
plt.imshow(test_img)

In [ ]:
test_img.shape

In [ ]:
test_img = cv2.resize(test_img, (256, 256))

In [ ]:
# making this as the input of the model

test_input = test_img.reshape((1, 256, 256, 3))

In [ ]:
model.predict(test_input)